# Exercise 30.1d solution


In [1]:
# --- Setup code from previous sub-exercises ---
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
# ----------------------------------------------

In [ ]:
import ipywidgets as widgets


def razumova_basic_widget(k_on=400, f=50, h=8, g=4):
    """Solve and plot the basic Razumova model with given parameters."""
    R_T = 1
    k_off = 50
    f_prime = 400
    h_prime = 6

    def rhs_widget(t, y):
        D, A_1, A_2 = y
        R_off = R_T - D - A_1 - A_2
        dD_dt = k_on * R_off + f_prime * A_1 + g * A_2 - (k_off + f) * D
        dA1_dt = f * D + h_prime * A_2 - (f_prime + h) * A_1
        dA2_dt = h * A_1 - (h_prime + g) * A_2
        return [dD_dt, dA1_dt, dA2_dt]

    t_span = (0, 10)
    t_eval = np.linspace(*t_span, 5000)
    sol = solve_ivp(rhs_widget, t_span, [0, 0, 0], t_eval=t_eval, method="RK45")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

    # State probabilities
    axes[0].plot(sol.t, sol.y[0], label=r"$D$")
    axes[0].plot(sol.t, sol.y[1], label=r"$A_1$")
    axes[0].plot(sol.t, sol.y[2], label=r"$A_2$")
    axes[0].set(
        xlabel="Time (s)",
        ylabel="State probability",
        title="State probabilities",
        ylim=(0, 1),
    )
    axes[0].legend()

    # Force development
    A_2_sol = sol.y[2]
    axes[1].plot(sol.t, A_2_sol, color="C3", label="Relative force")
    axes[1].set(
        xlabel="Time (s)",
        ylabel="Relative force ($A_2$)",
        xlim=(0, 1),
        ylim=(0, 1),
    )

    # k_dev calculation
    f_max = A_2_sol[-1]
    if f_max > 0:
        f_63 = (1 - 1 / np.e) * f_max
        idx = np.searchsorted(A_2_sol, f_63)
        if idx < len(sol.t):
            t_63 = sol.t[idx]
            axes[1].axhline(f_63, color="gray", linestyle="--", alpha=0.5)
            axes[1].axvline(t_63, color="gray", linestyle="--", alpha=0.5)
            axes[1].set_title(f"Force development ($k_{{dev}}$ = {1 / t_63:.1f} 1/s)")
        else:
            axes[1].set_title("Force development")
    else:
        axes[1].set_title("Force development")

    plt.show()


widgets.interact(
    razumova_basic_widget,
    k_on=widgets.FloatSlider(value=400, min=100, max=500, step=10, description=r"k_on"),
    f=widgets.FloatSlider(value=50, min=10, max=200, step=5, description="f"),
    h=widgets.FloatSlider(value=8, min=1, max=30, step=1, description="h"),
    g=widgets.FloatSlider(value=4, min=1, max=20, step=1, description="g"),
);

interactive(children=(FloatSlider(value=400.0, description='k_on', max=500.0, min=100.0, step=10.0), FloatSlid…

3. What happens to the steady-state $A_2$ value if you increase $k_{\mathrm{on}}$? What about $f$?

Increasing $k_{\mathrm{on}}$ shifts more RUs from $R_{\mathrm{off}}$ to $D$, making more sites available for XB attachment. This increases the steady-state $A_2$ value and the overall force. Increasing $f$ similarly boosts force by increasing the rate at which detached XBs ($D$) bind to actin and enter $A_1$.

The pattern across both exercises is the same: simple factual questions go right after the computation, parameter exploration questions go after the widget.
